__Delete this cell later__:

---

idea for structuring the text in notebook: We could do something like this:
 - Start by saying "we have done x, y & Z know ...", "based on the load data we can know do x..." and etc..
 - Do the next step/task.
 - Finish each thing with *Answer:* "We found that ..."

---

Things that needs check/changes:
- Idk if this requirements.txt is correct, if it is I thing its a good idea, this could also be done in a gitworkflow file, but maybe too much for now. 
- Check if source(s) and quotes are correct
---

## Imports need for notebook 

In [912]:
# Uncomment line 2 and run this cell to install the required packages for the project.
# pip install -r requirements.txt

In [913]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

In [914]:
# Only uncomment line 2 and run this cell if the imports have changed
# pip freeze > requirements.txt

## 1. Bussiness case / Problem statement

A cross the world, we all use cars, som places more then others, which leads to a lot cars being bought and sold. According to S&P Global, there was in <br><br>
&emsp;&emsp;*"April 2025 US auto sales in April expected to reach 1.49 million units [...]"* <br><br> source: [automotive-insights](https://www.spglobal.com/automotive-insights/en/blogs/2025/02/us-auto-sales-2025) (last visited: 29-04-2025). 
<br>

Where there is a lot of cars being sold, they have a possibility that they can end up at used car dealerships, which there are a large amount of according to [ibisworld](https://www.ibisworld.com/united-states/number-of-businesses/used-car-dealers/1004/) (last visited: 29-04-2025), where <br>

&emsp;&emsp;*"[...] 130,152 Used Car Dealers in the US businesses as of 2023, an decrease of -0.6% from 2022."*
<br>

Based on this we can wonder how can these used car dealerships price their cars? What factors are important and make weight in the pricing of a car? Can this be done more efficiently by using machine learning? And if so, how?

## 2. Data selection and preparation:

This section will include:
- x
- y


This exam project is based on the dataset(s): [Car Prices Dataset](https://www.kaggle.com/datasets/sidharth178/car-prices-dataset?select=train.csv) from kaggle. This contains two  datasets: test.csv and train.csv, where the test set dont contain price. These sets of data don't seem to have been pre cleaned. A thing to consider is that if the test.csv is to close to the train.csv, it might give some bias in the model and fit it to well, so we need to be careful with that.

### 2.1 Load data

In [915]:
base_path = '../data/'

df_test = pd.read_csv(f'{base_path}test.csv', sep=',', header=0)
df = pd.read_csv(f'{base_path}train.csv', sep=',', header=0)

We will load the data using pandas read_csv function, where we have specified the separator to separate by sep = ',' and with header = 0 to instruct that the first row of the csv file is a header, wich will become the columns in the dataframe.

After the data has been loaded, we verify that the data has been loadded properly by sampling 5 ranomd rows of the data, by using the sample function: 

In [916]:
df.sample(5)

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
5133,45804162,27065,836,HYUNDAI,Tucson,2010,Jeep,Yes,Diesel,2,102138 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4
1342,44531804,53941,891,MERCEDES-BENZ,GLA 250,2016,Jeep,Yes,Petrol,2.0 Turbo,87091 km,4.0,Tiptronic,Front,04-May,Left wheel,Grey,10
7623,45805545,19757,-,MERCURY,Mariner,2009,Jeep,Yes,Hybrid,2.5,261000 km,4.0,Automatic,Front,04-May,Left wheel,Silver,10
3829,45809948,47201,900,HONDA,Civic,2015,Sedan,Yes,Petrol,2.4,104867 km,4.0,Automatic,Front,04-May,Left wheel,Black,4
5092,44729405,20385,-,BMW,X5,2004,Jeep,Yes,Diesel,3.0 Turbo,230000 km,6.0,Tiptronic,4x4,>5,Left wheel,Grey,6


In [917]:
df_test.sample(5)

,ID,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags,Price
5728,45771220,1172,MERCEDES-BENZ,E 350,2011,Sedan,Yes,Diesel,3.5,230696 km,6,Automatic,4x4,04-May,Left wheel,Black,12,NaN
1077,45802321,934,HYUNDAI,H1,2015,Universal,Yes,Diesel,2.5,145240 km,4,Automatic,Front,04-May,Left wheel,Silver,4,NaN
6347,42994069,-,TOYOTA,ISIS,2007,Minivan,No,Petrol,2,220000 km,4,Tiptronic,Front,04-May,Right-hand drive,Golden,2,NaN
2913,45816120,-,CHRYSLER,Voyager,2003,Minivan,No,Diesel,2.5 Turbo,310000 km,6,Manual,Front,04-May,Left wheel,Grey,2,NaN
8196,45764923,1363,LEXUS,GX 460,2012,Jeep,Yes,Petrol,4.6,229334 km,8,Automatic,4x4,04-May,Left wheel,Black,0,NaN


In [918]:
df.shape

(19237, 18)

We can see that the data has been loaded properly. Additionally this process also helps us to quickly get a view and idea of the data we are going to work with. 

We can first of all see that the `ID` column is not need beacuse the datafram has its own bulid in index. 
Secondly we can see that the `Doors` column looks a bit strange, beacause it containes 3 opstions: '04-May', '02-Mar' or '>5', which couold be a mistake in the data, so this has to be checked more in detail. 
Thirdly we can see that when `Levy` don't have a value its just a '-'. 

### 2.2 Cleaning- and Preprocessing Data

In [919]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors             19237 non-null  object 
 15  Wheel             19237 non-null  object 
 16  Color             19237 non-null  object

We can see that there are no null values in the data; however we know that there are some columns like `Levy`, that have '-' instead of NaN.

In [920]:
df_dash_count = df['Levy'].where(df['Levy'] == '-').count()
df_test_dash_count = df_test['Levy'].where(df_test['Levy'] == '-').count()

print(f"Number of '-' in Levy in train set: {df_dash_count}")
print(f"Number of '-' in Levy in test set: {df_test_dash_count}")

Number of '-' in Levy in train set: 5819
Number of '-' in Levy in test set: 2454


In [921]:
levy_avg = np.sum(df['Levy'].where(df['Levy'] != '-').astype(float) / df.shape[0])
levy_test_avg = np.sum(df_test['Levy'].where(df_test['Levy'] != '-').astype(float) / df_test.shape[0])

print(f"Average Levy in train set: {levy_avg}")
print(f"Average Levy in test set: {levy_test_avg}")

df['Levy'] = df['Levy'].replace('-', int(levy_avg))
df_test['Levy'] = df_test['Levy'].replace('-', int(levy_test_avg))


Average Levy in train set: 632.5286687113374
Average Levy in test set: 644.672771376592


In [922]:
df['Doors'].value_counts()

Doors
04-May    18332
02-Mar      777
>5          128
Name: count, dtype: int64

In [923]:
df['Doors'] = df['Doors'].replace({
    '04-May': '4',
    '02-Mar': '2',
    '>5': '5'
})

In [924]:
df['Mileage'] = df['Mileage'].str.replace(' km', '')
df_test['Mileage'] = df_test['Mileage'].str.replace(' km', '')

df.rename(columns={'Mileage': 'Mileage_km'}, inplace=True)
df_test.rename(columns={'Mileage': 'Mileage_km'}, inplace=True)

In [925]:
to_numeric = ['Prod. year', 'Mileage_km', 'Levy', 'Doors']

df[to_numeric] = df[to_numeric].apply(pd.to_numeric, errors='coerce')
df_test[to_numeric] = df_test[to_numeric].apply(pd.to_numeric, errors='coerce')

In [926]:
df.drop(columns=['ID'], inplace=True)
df_test.drop(columns=['ID'], inplace=True)

As stated previously, we remove `ID` because we don't need two indexes.

#### 2.2.x Verifying data after cleaning

Just in case, we check if the values have been drop, have missing values and that we have the right data types, so we don't end with unexpected values in the data. We do this as we did after loading the data, by randomly sampling 5 rows from the datasets and using the info function to see null values and data types.

In [927]:
df.sample(5)

,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage_km,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
14682,470,1493,LEXUS,RX 350,2016,Jeep,Yes,Petrol,3.5,118133,6.0,Automatic,4x4,4,Left wheel,White,12
5269,1019,1537,TOYOTA,Highlander,2009,Sedan,Yes,Hybrid,3.3,318358,6.0,Automatic,4x4,4,Left wheel,Black,12
6058,10976,632,OPEL,Combo TDI,2006,Goods wagon,No,Diesel,1.6,200000,4.0,Manual,Front,4,Left wheel,Blue,6
18769,9565,2377,DODGE,Ramcharger,2016,Sedan,Yes,Petrol,5.7,39013,8.0,Automatic,Rear,4,Left wheel,Black,12
6451,78829,831,HONDA,Civic,2018,Hatchback,Yes,Petrol,1.5,2000,4.0,Automatic,Front,4,Left wheel,Black,4


In [928]:
df_test.sample(5)

,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage_km,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags,Price
2423,629,HYUNDAI,Elantra,2015,Sedan,Yes,Petrol,1.6,67590,4,Automatic,Front,NaN,Left wheel,Silver,4,NaN
4996,644,TOYOTA,Camry,2014,Sedan,Yes,Petrol,2.5,128000,4,Tiptronic,Front,NaN,Left wheel,Black,0,NaN
8147,642,HYUNDAI,Tucson,2012,Jeep,Yes,Diesel,2,179099,4,Automatic,Front,NaN,Left wheel,Black,4,NaN
2728,503,TOYOTA,Prius C,2012,Hatchback,No,Hybrid,1.5,180000,4,Automatic,Front,NaN,Left wheel,White,6,NaN
4518,644,VAZ,21099,1999,Sedan,Yes,CNG,3.5,0,6,Automatic,4x4,NaN,Left wheel,Black,2,NaN


In [929]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Price             19237 non-null  int64  
 1   Levy              19237 non-null  int64  
 2   Manufacturer      19237 non-null  object 
 3   Model             19237 non-null  object 
 4   Prod. year        19237 non-null  int64  
 5   Category          19237 non-null  object 
 6   Leather interior  19237 non-null  object 
 7   Fuel type         19237 non-null  object 
 8   Engine volume     19237 non-null  object 
 9   Mileage_km        19237 non-null  int64  
 10  Cylinders         19237 non-null  float64
 11  Gear box type     19237 non-null  object 
 12  Drive wheels      19237 non-null  object 
 13  Doors             19237 non-null  int64  
 14  Wheel             19237 non-null  object 
 15  Color             19237 non-null  object 
 16  Airbags           19237 non-null  int64 

We can know comfirm that the values have infact been droped, have the right data types and that we have no missing values.

Lastly we save the cleaned data to a new csv file ...

In [930]:
# (maybe) save to new csv files

## 3 Data exploration and visualization:

This section will include:
- x
- y



--- 
Look at the data and try to understand it. What are the most important features?

Make some visualizations to understand the data better, by showing the distribution of the features and the target variable.

Try to find correlations between the features and the target variable, by making scatter plots and correlation matrices.

etc ...

## 4. Model Selection and training:

This section will include:
- x
- y

### 4.1 The selected model(s)

We have selected the following models for this project:
- Multiple Linear Regression
- Model 2

The reason for the selected models is as follows:
- Multiple Linear Regression: ..
- Model 2: This model is selected because ...

We are going to be using the following error metrics to evalaute the models:
- x
- y
- z

Additionaly we are going to be useing the grid search approach for ...

### 4.2 Model training and validation

Based on the selected models we have trained the models using the training data. The training process is as follows:

"describe the training process here, that we are going to use and why that makes sence"

### 4.3 Model evaluation